In [ ]:
from pathlib import Path
import pydicom
import numpy as np
import pandas as pd
from tqdm import tqdm

DATA_ROOT = Path(r"D:\TCIA\nsclc_raw\nsclc_radiomics")

patient_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])

print("Toplam hasta klasörü:", len(patient_dirs))
print("İlk 10 hasta:")
for p in patient_dirs[:10]:
    print(p.name)

Toplam hasta klasörü: 422
İlk 10 hasta:
LUNG1-001
LUNG1-002
LUNG1-003
LUNG1-004
LUNG1-005
LUNG1-006
LUNG1-007
LUNG1-008
LUNG1-009
LUNG1-010


In [ ]:
excluded_cases = {"LUNG1-014", "LUNG1-021", "LUNG1-085"}

patient_dirs = [p for p in patient_dirs if p.name not in excluded_cases]

print("Hariç tutulanlardan sonra hasta sayısı:", len(patient_dirs))

Hariç tutulanlardan sonra hasta sayısı: 419


In [ ]:
def get_dcm_files(patient_path):
    return list(patient_path.rglob("*.dcm"))

In [ ]:
def split_ct_and_seg_files(dcm_files):
    ct_files = []
    seg_files = []

    for f in dcm_files:
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            modality = getattr(ds, "Modality", None)

            if modality == "CT":
                ct_files.append(f)
            elif modality == "SEG":
                seg_files.append(f)
        except:
            pass

    return ct_files, seg_files

In [ ]:
def find_tumor_segment_number(seg_ds):
    if not hasattr(seg_ds, "SegmentSequence"):
        return None

    for seg_item in seg_ds.SegmentSequence:
        label = str(getattr(seg_item, "SegmentLabel", "")).lower()
        desc = str(getattr(seg_item, "SegmentDescription", "")).lower()

        if (
            "neoplasm" in label
            or "gtv" in desc
            or "tumor" in label
            or "tumour" in label
            or "primary" in label
        ):
            return int(seg_item.SegmentNumber)

    return None

In [ ]:
def inspect_patient(patient_path):
    result = {
        "patient_id": patient_path.name,
        "num_dcm_files": 0,
        "num_ct_files": 0,
        "num_seg_files": 0,
        "tumor_segment_number": None,
        "status": "unknown",
        "note": ""
    }

    try:
        dcm_files = get_dcm_files(patient_path)
        result["num_dcm_files"] = len(dcm_files)

        ct_files, seg_files = split_ct_and_seg_files(dcm_files)
        result["num_ct_files"] = len(ct_files)
        result["num_seg_files"] = len(seg_files)

        if len(ct_files) == 0:
            result["status"] = "fail"
            result["note"] = "No CT files"
            return result

        if len(seg_files) == 0:
            result["status"] = "fail"
            result["note"] = "No SEG file"
            return result

        seg_ds = pydicom.dcmread(str(seg_files[0]), stop_before_pixels=False)
        tumor_segment_number = find_tumor_segment_number(seg_ds)
        result["tumor_segment_number"] = tumor_segment_number

        if tumor_segment_number is None:
            result["status"] = "fail"
            result["note"] = "Tumor segment not found"
            return result

        result["status"] = "ok"
        result["note"] = "Ready"

    except Exception as e:
        result["status"] = "error"
        result["note"] = str(e)

    return result

In [ ]:
sample_results = []

for patient_path in patient_dirs[:10]:
    res = inspect_patient(patient_path)
    sample_results.append(res)

sample_df = pd.DataFrame(sample_results)
sample_df

,patient_id,num_dcm_files,num_ct_files,num_seg_files,tumor_segment_number,status,note
0,LUNG1-001,136,134,1,1,ok,Ready
1,LUNG1-002,113,111,1,2,ok,Ready
2,LUNG1-003,109,107,1,2,ok,Ready
3,LUNG1-004,116,114,1,2,ok,Ready
4,LUNG1-005,93,91,1,2,ok,Ready
5,LUNG1-006,116,114,1,2,ok,Ready
6,LUNG1-007,131,129,1,2,ok,Ready
7,LUNG1-008,116,114,1,2,ok,Ready
8,LUNG1-009,107,105,1,2,ok,Ready
9,LUNG1-010,93,91,1,2,ok,Ready


In [ ]:
def build_ct_volume(ct_files):
    def get_z(dcm_path):
        ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        ipp = getattr(ds, "ImagePositionPatient", None)
        return float(ipp[2]) if ipp is not None else 0.0

    ct_files_sorted = sorted(ct_files, key=get_z)

    ct_slices = [pydicom.dcmread(str(f)) for f in ct_files_sorted]
    volume = np.stack([ds.pixel_array for ds in ct_slices], axis=0)

    slope = float(getattr(ct_slices[0], "RescaleSlope", 1.0))
    intercept = float(getattr(ct_slices[0], "RescaleIntercept", 0.0))

    volume_hu = volume.astype(np.float32) * slope + intercept

    return volume_hu, ct_files_sorted

In [ ]:
def extract_tumor_mask(seg_ds, segment_number):
    seg_array = seg_ds.pixel_array
    pffgs = seg_ds.PerFrameFunctionalGroupsSequence

    segment_numbers = [
        int(frame.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
        for frame in pffgs
    ]

    tumor_indices = [
        i for i, s in enumerate(segment_numbers)
        if s == segment_number
    ]

    tumor_mask = seg_array[tumor_indices]
    return tumor_mask

In [ ]:
def process_patient(patient_path):
    dcm_files = get_dcm_files(patient_path)
    ct_files, seg_files = split_ct_and_seg_files(dcm_files)

    if len(ct_files) == 0:
        return {"status": "fail", "note": "No CT"}
    if len(seg_files) == 0:
        return {"status": "fail", "note": "No SEG"}

    try:
        seg_ds = pydicom.dcmread(str(seg_files[0]))
        tumor_segment = find_tumor_segment_number(seg_ds)

        if tumor_segment is None:
            return {"status": "fail", "note": "Tumor segment not found"}

        volume_hu, ct_files_sorted = build_ct_volume(ct_files)
        tumor_mask = extract_tumor_mask(seg_ds, tumor_segment)

        if volume_hu.shape != tumor_mask.shape:
            return {
                "status": "fail",
                "note": f"Shape mismatch: image {volume_hu.shape}, mask {tumor_mask.shape}"
            }

        if tumor_mask.sum() == 0:
            return {
                "status": "fail",
                "note": "Empty tumor mask"
            }

        return {
            "status": "ok",
            "note": "Processed",
            "image_shape": volume_hu.shape,
            "mask_shape": tumor_mask.shape,
            "mask_sum": int(tumor_mask.sum())
        }

    except Exception as e:
        return {"status": "error", "note": str(e)}

In [ ]:
process_results = []

for patient_path in patient_dirs[:5]:
    result = process_patient(patient_path)
    process_results.append({
        "patient_id": patient_path.name,
        **result
    })

process_df = pd.DataFrame(process_results)
process_df

,patient_id,status,note,image_shape,mask_shape,mask_sum
0,LUNG1-001,ok,Processed,"(134, 512, 512)","(134, 512, 512)",54639
1,LUNG1-002,ok,Processed,"(111, 512, 512)","(111, 512, 512)",125531
2,LUNG1-003,ok,Processed,"(107, 512, 512)","(107, 512, 512)",12159
3,LUNG1-004,ok,Processed,"(114, 512, 512)","(114, 512, 512)",29521
4,LUNG1-005,ok,Processed,"(91, 512, 512)","(91, 512, 512)",29176


In [ ]:
process_results_20 = []

for patient_path in patient_dirs[5:20]:
    result = process_patient(patient_path)
    process_results_20.append({
        "patient_id": patient_path.name,
        **result
    })

process_df_20 = pd.DataFrame(process_results_20)
process_df_20

,patient_id,status,note,image_shape,mask_shape,mask_sum
0,LUNG1-006,ok,Processed,"(114, 512, 512)","(114, 512, 512)",27358
1,LUNG1-007,ok,Processed,"(129, 512, 512)","(129, 512, 512)",4367
2,LUNG1-008,ok,Processed,"(114, 512, 512)","(114, 512, 512)",14272
3,LUNG1-009,ok,Processed,"(105, 512, 512)","(105, 512, 512)",34445
4,LUNG1-010,ok,Processed,"(91, 512, 512)","(91, 512, 512)",5634
5,LUNG1-011,ok,Processed,"(88, 512, 512)","(88, 512, 512)",27882
6,LUNG1-012,ok,Processed,"(109, 512, 512)","(109, 512, 512)",90600
7,LUNG1-013,ok,Processed,"(134, 512, 512)","(134, 512, 512)",5088
8,LUNG1-015,ok,Processed,"(117, 512, 512)","(117, 512, 512)",1636
9,LUNG1-016,ok,Processed,"(82, 512, 512)","(82, 512, 512)",39731


In [ ]:
OUTPUT_DIR = Path("processed")

images_dir = OUTPUT_DIR / "images"
masks_dir = OUTPUT_DIR / "masks"

images_dir.mkdir(parents=True, exist_ok=True)
masks_dir.mkdir(parents=True, exist_ok=True)

print("Klasörler hazır")

Klasörler hazır


In [ ]:
patient = patient_dirs[0]

result = process_patient(patient)

if result["status"] == "ok":
    volume_hu, tumor_mask = None, None
    
    # yeniden al (çünkü process_patient sadece metadata döndürüyor)
    dcm_files = get_dcm_files(patient)
    ct_files, seg_files = split_ct_and_seg_files(dcm_files)
    
    seg_ds = pydicom.dcmread(str(seg_files[0]))
    tumor_segment = find_tumor_segment_number(seg_ds)
    
    volume_hu, _ = build_ct_volume(ct_files)
    tumor_mask = extract_tumor_mask(seg_ds, tumor_segment)

    np.save(images_dir / f"{patient.name}.npy", volume_hu)
    np.save(masks_dir / f"{patient.name}.npy", tumor_mask)

    print("Kaydedildi:", patient.name)

Kaydedildi: LUNG1-001


In [ ]:
img = np.load(images_dir / "LUNG1-001.npy")
mask = np.load(masks_dir / "LUNG1-001.npy")

print(img.shape, mask.shape)

(134, 512, 512) (134, 512, 512)


In [21]:
saved = []
failed = []

for patient in tqdm(patient_dirs):
    result = process_patient(patient)

    if result["status"] != "ok":
        failed.append((patient.name, result["note"]))
        continue

    try:
        dcm_files = get_dcm_files(patient)
        ct_files, seg_files = split_ct_and_seg_files(dcm_files)

        seg_ds = pydicom.dcmread(str(seg_files[0]))
        tumor_segment = find_tumor_segment_number(seg_ds)

        volume_hu, _ = build_ct_volume(ct_files)
        tumor_mask = extract_tumor_mask(seg_ds, tumor_segment)

        np.save(images_dir / f"{patient.name}.npy", volume_hu)
        np.save(masks_dir / f"{patient.name}.npy", tumor_mask)

        # 🔥 RAM temizle
        del volume_hu, tumor_mask

        saved.append(patient.name)

    except Exception as e:
        failed.append((patient.name, str(e)))

print("Saved:", len(saved))
print("Failed:", len(failed))

 26%|██▌       | 107/419 [13:56<40:40,  7.82s/it]  


KeyboardInterrupt: 

In [22]:
from pathlib import Path

images_dir = Path("processed/images")
masks_dir = Path("processed/masks")

image_files = sorted(images_dir.glob("*.npy"))
mask_files = sorted(masks_dir.glob("*.npy"))

print("Image sayısı:", len(image_files))
print("Mask sayısı :", len(mask_files))
print("İlk 5 image:", [f.name for f in image_files[:5]])
print("İlk 5 mask :", [f.name for f in mask_files[:5]])

Image sayısı: 106
Mask sayısı : 106
İlk 5 image: ['LUNG1-001.npy', 'LUNG1-002.npy', 'LUNG1-003.npy', 'LUNG1-004.npy', 'LUNG1-005.npy']
İlk 5 mask : ['LUNG1-001.npy', 'LUNG1-002.npy', 'LUNG1-003.npy', 'LUNG1-004.npy', 'LUNG1-005.npy']


In [23]:
image_names = {f.stem for f in image_files}
mask_names = {f.stem for f in mask_files}

print("Sadece image'da olan:", len(image_names - mask_names))
print("Sadece mask'te olan :", len(mask_names - image_names))
print("Ortak dosya sayısı   :", len(image_names & mask_names))

Sadece image'da olan: 0
Sadece mask'te olan : 0
Ortak dosya sayısı   : 106


In [24]:
from pathlib import Path
import numpy as np
import pandas as pd

images_dir = Path("processed/images")
masks_dir = Path("processed/masks")

records = []

for img_path in sorted(images_dir.glob("*.npy")):
    patient_id = img_path.stem
    mask_path = masks_dir / f"{patient_id}.npy"

    if not mask_path.exists():
        continue

    img = np.load(img_path, mmap_mode="r")
    mask = np.load(mask_path, mmap_mode="r")

    records.append({
        "patient_id": patient_id,
        "image_path": str(img_path),
        "mask_path": str(mask_path),
        "image_shape": tuple(img.shape),
        "mask_shape": tuple(mask.shape),
        "num_slices": img.shape[0],
        "mask_sum": int(mask.sum()),
        "num_mask_slices": int((mask.reshape(mask.shape[0], -1).sum(axis=1) > 0).sum())
    })

index_df = pd.DataFrame(records)
index_df.head()

,patient_id,image_path,mask_path,image_shape,mask_shape,num_slices,mask_sum,num_mask_slices
0,LUNG1-001,processed\images\LUNG1-001.npy,processed\masks\LUNG1-001.npy,"(134, 512, 512)","(134, 512, 512)",134,54639,21
1,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,"(111, 512, 512)","(111, 512, 512)",111,125531,26
2,LUNG1-003,processed\images\LUNG1-003.npy,processed\masks\LUNG1-003.npy,"(107, 512, 512)","(107, 512, 512)",107,12159,17
3,LUNG1-004,processed\images\LUNG1-004.npy,processed\masks\LUNG1-004.npy,"(114, 512, 512)","(114, 512, 512)",114,29521,36
4,LUNG1-005,processed\images\LUNG1-005.npy,processed\masks\LUNG1-005.npy,"(91, 512, 512)","(91, 512, 512)",91,29176,24


In [25]:
print("Toplam hasta:", len(index_df))
print(index_df[["num_slices", "mask_sum", "num_mask_slices"]].describe())

Toplam hasta: 106
       num_slices       mask_sum  num_mask_slices
count  106.000000     106.000000       106.000000
mean   122.320755   29947.632075        18.764151
std     33.803507   36547.396999        10.830778
min     82.000000     290.000000         3.000000
25%    104.000000    3925.250000        10.250000
50%    114.000000   18662.000000        17.000000
75%    134.000000   40259.000000        24.000000
max    297.000000  230950.000000        51.000000


In [26]:
index_df.to_csv("processed/nsclc_index.csv", index=False)
print("Kaydedildi: processed/nsclc_index.csv")

Kaydedildi: processed/nsclc_index.csv


In [29]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    index_df["patient_id"].tolist(),
    test_size=0.2,
    random_state=42
)

train_df = index_df[index_df["patient_id"].isin(train_ids)].reset_index(drop=True)
val_df = index_df[index_df["patient_id"].isin(val_ids)].reset_index(drop=True)

print("Train hasta sayısı:", len(train_df))
print("Val hasta sayısı  :", len(val_df))

Train hasta sayısı: 84
Val hasta sayısı  : 22


In [30]:
train_df.to_csv("processed/train_patients.csv", index=False)
val_df.to_csv("processed/val_patients.csv", index=False)

In [31]:
slice_records = []

for _, row in train_df.iterrows():
    patient_id = row["patient_id"]
    img = np.load(row["image_path"], mmap_mode="r")
    mask = np.load(row["mask_path"], mmap_mode="r")

    mask_per_slice = mask.reshape(mask.shape[0], -1).sum(axis=1)
    positive_slices = np.where(mask_per_slice > 0)[0]

    for s in positive_slices:
        slice_records.append({
            "patient_id": patient_id,
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
            "slice_idx": int(s),
            "split": "train"
        })

for _, row in val_df.iterrows():
    img = np.load(row["image_path"], mmap_mode="r")
    mask = np.load(row["mask_path"], mmap_mode="r")

    mask_per_slice = mask.reshape(mask.shape[0], -1).sum(axis=1)
    positive_slices = np.where(mask_per_slice > 0)[0]

    for s in positive_slices:
        slice_records.append({
            "patient_id": row["patient_id"],
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
            "slice_idx": int(s),
            "split": "val"
        })

slice_df = pd.DataFrame(slice_records)
slice_df.head()

,patient_id,image_path,mask_path,slice_idx,split
0,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,28,train
1,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,29,train
2,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,30,train
3,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,31,train
4,LUNG1-002,processed\images\LUNG1-002.npy,processed\masks\LUNG1-002.npy,32,train


In [32]:
print("Toplam pozitif slice:", len(slice_df))
print(slice_df["split"].value_counts())

Toplam pozitif slice: 1989
split
train    1627
val       362
Name: count, dtype: int64


In [33]:
slice_df.to_csv("processed/slice_index_positive_only.csv", index=False)

In [34]:
print("Train hasta sayısı:", len(train_df))
print("Val hasta sayısı  :", len(val_df))

Train hasta sayısı: 84
Val hasta sayısı  : 22
